# 4. Population dynamics and phenotypic switching

Migration models often include birth, death and transitions
between phenotypes. These processes keep different quantities, and
BioLGCA treats them as different kinds of interaction.

**Learning objectives**

- choose the order of birth/death, phenotype switching and reorientation;
- verify the complete-state conservation rule `s -> s'`;
- plot total population and phenotype fractions; and
- build the go-or-grow model from migrating and resting cells.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from lgca.model import (
    AnalysisSpec,
    Description,
    ModelSpec,
    SpaceSpec,
    StateSpec,
    TimeSpec,
    run_model,
)
from lgca.pipeline import (
    BirthDeathSpec,
    InteractionPipelineSpec,
    PhenotypeSwitchSpec,
    ReorientationSpec,
    ReorientationTermSpec,
)
from lgca.simulation import DensityRecorder, NodeRecorder, PopulationRecorder


## Conservation means sampling the full channel state

At one node, `s` is the complete `(species, channels)` state. A
particle-number-conserving phenotype interaction samples one
admissible state `s'` with the same total occupancy. It must not
perform independent channel writes that could collide, create or
remove particles.

We first isolate switching for one time step and turn propagation
off. This makes the invariant directly observable.


In [ ]:
switch_nodes = np.zeros((4, 4, 2, 5), dtype=bool)
switch_nodes[1, 1, 0, 0] = True
switch_nodes[1, 2, 0, 4] = True
switch_nodes[2, 1, 1, 2] = True

switch_only_spec = ModelSpec(
    description=Description(title="Atomic phenotype switch"),
    space=SpaceSpec(geometry="square", boundary="periodic"),
    state=StateSpec(nodes=switch_nodes, restchannels=1, n_species=2),
    time=TimeSpec(steps=1, seed=41),
    dynamics=InteractionPipelineSpec(
        operators=[
            PhenotypeSwitchSpec(
                name="phenotype_switch",
                parameters={"rates": [[0.0, 1.0], [0.0, 0.0]]},
            )
        ],
        propagation=False,
    ),
    analysis=AnalysisSpec(observers=[NodeRecorder(), DensityRecorder()]),
)

switch_only = run_model(switch_only_spec, showprogress=False)
before = switch_only.lgca.nodes_t[0]
after = switch_only.lgca.nodes_t[1]
assert before.sum() == after.sum()
print("particles by species before:", before.sum(axis=(0, 1, 3)))
print("particles by species after: ", after.sum(axis=(0, 1, 3)))
print("total particles conserved:", before.sum(), after.sum())


The forced switch changes species identity while preserving total
particle number. Random walk, alignment and chemotaxis in
volume-exclusion models use the same full-state principle for their
particle-conserving channel transitions.


## A complete sequential biological pipeline

Birth/death may change total population, phenotype switching
redistributes that population between species, reorientation
changes channels without changing particle number, and propagation
moves the selected channels. The operators run in the order they
are listed, and the schedule below shows it.


In [ ]:
population_spec = ModelSpec(
    description=Description(title="Growing and switching population"),
    space=SpaceSpec(
        geometry="square",
        dims=(12, 12),
        boundary="periodic",
    ),
    state=StateSpec(
        density=0.2,
        restchannels=1,
        n_species=2,
    ),
    time=TimeSpec(steps=20, seed=44),
    dynamics=InteractionPipelineSpec(
        operators=[
            BirthDeathSpec(
                name="birth_death",
                parameters={
                    "birth_rate": [0.03, 0.01],
                    "death_rate": [0.005, 0.005],
                },
            ),
            PhenotypeSwitchSpec(
                name="phenotype_switch",
                parameters={
                    "rates": [[0.0, 0.08], [0.03, 0.0]],
                },
            ),
            ReorientationSpec(
                terms=[ReorientationTermSpec(name="random_walk")],
            ),
        ],
    ),
    analysis=AnalysisSpec(
        observers=[NodeRecorder(), DensityRecorder(), PopulationRecorder()],
    ),
)

population_result = run_model(population_spec, showprogress=False)
print(population_result.metadata["schedule"])


In [ ]:
species_populations = population_result.lgca.dens_t.sum(axis=(1, 2))
species_fractions = species_populations / species_populations.sum(axis=1, keepdims=True)
steps = population_result.lgca.n_steps

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), constrained_layout=True)
axes[0].plot(steps, population_result.lgca.n_t, color="black")
axes[0].set(xlabel="time step", ylabel="total population")
axes[1].plot(steps, species_fractions[:, 0], label="phenotype 0")
axes[1].plot(steps, species_fractions[:, 1], label="phenotype 1")
axes[1].set(xlabel="time step", ylabel="population fraction", ylim=(0, 1))
axes[1].legend()
plt.show()
plt.close(fig)


Total population changes because birth/death is present. Phenotype
fractions can change through both differential birth rates and the
switch matrix. Interpreting either mechanism requires a control in
which the other is removed.


## Go-or-grow

In go-or-grow models, cells either migrate or rest and divide, and
switch between the two depending on how crowded their node is. A cell
in a velocity channel moves, a cell in a rest channel rests. A time
step is the sequence

1. `go_or_rest`: a moving cell starts resting with probability
   $(1 + \tanh(\kappa(\rho - \theta)))/2$ and a resting cell starts
   moving with the complementary probability, where $\rho$ is the
   fraction of occupied channels. The cells stay at their node and only
   change channels, so this is a reorientation;
2. `go_or_grow.growth`: cells die, and resting cells divide into free
   rest channels;
3. `random_walk`: moving cells pick random velocity channels;

followed by propagation. This is the go-or-grow model of earlier
versions: `get_lgca(interaction="go_or_grow")` runs these three rules. We compare $\kappa = 4$, where crowded cells rest, with
$\kappa = -4$, where crowded cells migrate.


In [ ]:
dims = (40, 40)
nodes = np.zeros(dims + (12,), dtype=bool)  # 6 velocity and 6 rest channels
nodes[15:25, 15:25, :] = True  # a colony of 10 x 10 full nodes


def go_or_grow_spec(kappa):
    return ModelSpec(
        description=Description(title=f"Go-or-grow, kappa = {kappa}"),
        space=SpaceSpec(geometry="hex", dims=dims, boundary="periodic"),
        state=StateSpec(nodes=nodes, restchannels=6),
        time=TimeSpec(steps=60, seed=47),
        dynamics=InteractionPipelineSpec(
            operators=[
                {"name": "go_or_rest", "parameters": {"kappa": kappa, "theta": 0.75}},
                {"name": "go_or_grow.growth", "parameters": {"r_b": 0.2, "r_d": 0.01}},
                {"name": "random_walk", "parameters": {"channels": "velocity"}},
            ],
        ),
        analysis=AnalysisSpec(observers=[NodeRecorder(), PopulationRecorder()]),
    )


go_or_grow = {kappa: run_model(go_or_grow_spec(kappa), showprogress=False) for kappa in (4.0, -4.0)}
print(go_or_grow[4.0].metadata["schedule"])


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5), sharey=True, constrained_layout=True)
for axis, (kappa, result) in zip(axes, go_or_grow.items()):
    history = result.lgca.nodes_t  # time, x, y, channels
    moving = history[..., :6].sum(axis=(1, 2, 3))
    resting = history[..., 6:].sum(axis=(1, 2, 3))
    steps = result.lgca.nodes_steps
    axis.plot(steps, moving, label="moving")
    axis.plot(steps, resting, label="resting")
    axis.plot(steps, moving + resting, color="black", label="total")
    axis.set(title=f"kappa = {kappa}", xlabel="time step")
axes[0].set_ylabel("cells")
axes[0].legend()
plt.show()
plt.close(fig)


With $\kappa = 4$, the total barely changes over 60 steps: as the
colony spreads out, its nodes become less crowded, resting cells start
migrating, and migrating cells do not divide. Smaller colonies shrink,
an Allee effect that emerges from the switching rule (the `go_or_grow`
example in the gallery starts from a single node). With
$\kappa = -4$, cells on sparse nodes rest and divide, and the
population more than triples.

The same model can also be written with two species, migrating cells
(species 0) and resting cells (species 1): the switch is then a
phenotype switch, `go_or_grow.switch`, and recorders count the two
phenotypes as species. The docstring of `lgca.builtin_rules` shows the
pipeline.


## Exercises

1. Set the switch rates to zero. Which changes remain in phenotype
   fractions and why?
2. Give both species the same birth rate and isolate switching.
3. Reverse the two off-diagonal switch rates and predict the final
   phenotype balance before running the model.
4. In go-or-grow, give resting cells a lower death rate with the
   parameter `r_d_resting` of `go_or_grow.growth`. How does the
   smallest colony that still grows change?
5. Set `when_full="reject"` in `go_or_rest` and `go_or_grow.growth`, so
   that every cell tries to switch or divide on its own. Compare the
   growth curves with the original rule.
